# 08 — Strength Progression Modeling V1

## Objective, dataset origin, and eligibility

Predict the representative maximum working weight in the next strictly later completed session for the same Exercise Library exercise or Custom Machine. Histories require four prior same-source sessions. Completed sessions, non-warmup weighted sets, and per-session maximum load are authoritative.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from metaform_ds.config import DataScienceConfig
from metaform_ds.modeling.strength import StrengthModelConfig, build_strength_prediction_examples, evaluate_strength_model, render_strength_evaluation_report
from metaform_ds.utils.notebooks import load_notebook_data
settings = DataScienceConfig.from_env()
data = load_notebook_data(settings)
origin = 'SYNTHETIC' if data.mode == 'SYNTHETIC' else 'REAL'
print(f'DATA ORIGIN: {origin}')
if origin == 'SYNTHETIC': print('SYNTHETIC PIPELINE VALIDATION ONLY — real evaluation remains pending.')
else: print('REAL READ-ONLY PSEUDONYMIZED EVALUATION.')

## Target construction, features, and leakage controls

Each row uses only current-and-earlier history from its exact anonymous user/source tuple. The next strictly later source session supplies the label. Features contain causal lags, rolling load/volume, elapsed time, working-set context, source type, and safe standard metadata. Raw IDs and custom-machine names are excluded. Preprocessing is fit inside expanding temporal folds whose training labels are already observed.

In [ ]:
config = StrengthModelConfig()
targets = build_strength_prediction_examples(data.strength, config)
print(f'Model version: {config.model_version}; minimum prior sessions: {config.minimum_prior_sessions}')
print(targets.eligibility_summary.to_string(index=False))
print(targets.exclusion_reasons.to_string(index=False))

## Baselines, Linear/Ridge, and temporal backtesting

Last Session predicts no change. Recent Trend projects a recent same-source session-index slope and falls back safely. Linear and Ridge use the same fold-local preprocessing. MAE in kilograms is primary; RMSE, safe R², prediction count, and direction accuracy are secondary.

In [ ]:
evaluation = evaluate_strength_model(data.strength, data_origin=origin, config=config)
print(f'Status: {evaluation.status}; predictions: {len(evaluation.predictions)}; folds: {len(evaluation.folds)}')
print(evaluation.comparison.to_string(index=False))
print(f'Best baseline: {evaluation.best_baseline}; preferred: {evaluation.preferred_method}; recommendation: {evaluation.recommendation}')

In [ ]:
predictions = evaluation.predictions
if predictions.empty:
    print('Insufficient data for actual-versus-predicted chart.')
else:
    view = predictions.sort_values('target_date')
    fig, ax = plt.subplots()
    ax.scatter(view['actual_next_max_weight_kg'], view['ridge_prediction_kg'], alpha=0.7)
    bounds = [min(view['actual_next_max_weight_kg'].min(), view['ridge_prediction_kg'].min()), max(view['actual_next_max_weight_kg'].max(), view['ridge_prediction_kg'].max())]
    ax.plot(bounds, bounds, linestyle='--', color='black')
    ax.set(title='Actual vs Ridge next-session maximum', xlabel='Actual kg', ylabel='Predicted kg')
    plt.show()

In [ ]:
if not evaluation.comparison.empty:
    ordered = evaluation.comparison.sort_values('mae_kg')
    fig, ax = plt.subplots()
    ax.bar(ordered['method'], ordered['mae_kg'])
    ax.set(title='Next-session MAE by method', xlabel='Method', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=20)
    plt.show()

## Temporal stability, segment analysis, and error analysis

The following aggregate tables examine fold stability and differences by source type, history length, and past cadence. Tiny groups are marked insufficient and are not interpreted as evidence.

In [ ]:
print('Temporal stability')
print(evaluation.stability.to_string(index=False))
print('Source type')
print(evaluation.source_type_metrics.to_string(index=False))
print('History length')
print(evaluation.history_metrics.to_string(index=False))
print('Cadence')
print(evaluation.cadence_metrics.to_string(index=False))

In [ ]:
if not evaluation.fold_metrics.empty:
    ridge_folds = evaluation.fold_metrics[evaluation.fold_metrics['method'].eq('Ridge Regression')]
    fig, ax = plt.subplots()
    ax.plot(ridge_folds['fold_id'], ridge_folds['mae_kg'], marker='o')
    ax.set(title='Ridge MAE across temporal folds', xlabel='Fold', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=90)
    plt.show()

In [ ]:
if not evaluation.source_type_metrics.empty:
    segment = evaluation.source_type_metrics[evaluation.source_type_metrics['method'].eq(evaluation.preferred_method)]
    fig, ax = plt.subplots()
    ax.bar(segment['source_type'], segment['mae_kg'])
    ax.set(title='MAE by source type', xlabel='Source type', ylabel='MAE (kg)')
    plt.show()

## Model-selection conclusion, limitations, and production recommendation

ML is not assumed to win. Promotion requires meaningful MAE improvement over the best baseline, sufficient predictions, and temporal stability. Sparse histories, irregular intent, deload context, substitutions, and machine differences remain limitations. Synthetic metrics validate mechanics only. No model is persisted or integrated.

In [ ]:
report = render_strength_evaluation_report(evaluation)
artifact_dir = settings.artifact_dir
if not artifact_dir.is_absolute() and Path.cwd().name == 'notebooks': artifact_dir = Path('..') / artifact_dir
artifact_dir.mkdir(parents=True, exist_ok=True)
report_path = artifact_dir / 'strength_progression_v1_evaluation.md'
report_path.write_text(report, encoding='utf-8')
print(f'Ignored aggregate report: {report_path}')
print(report)